# 66 — P10.7 SPIDER: dataset por nivel discal y crops 2.5D T1/T2

    **Objetivo:** transformar imágenes, máscaras y gradaciones de SPIDER en un manifest reproducible
    por paciente/nivel discal. Se generan crops 2.5D independientes para T1 y T2.

    Este notebook **no entrena** y no abre el contenido del `internal_test` para análisis de etiquetas.


> **Gobernanza obligatoria**
>
> - No reentrena estenosis central, foraminal ni subarticular: esas tareas P10.6 ya tienen checkpoints.
> - No accede al test oculto de SPIDER.
> - No usa el `internal_test` para seleccionar modelo o ajustar hiperparámetros.
> - Antes de escribir resultados audita notebooks, manifests, resultados, modelos y todos los `.pt`.
> - Si detecta un export final/frozen previo de P10.7, aborta.
> - La salida es de investigación, requiere revisión profesional y no constituye diagnóstico clínico.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!pip -q install SimpleITK>=2.3.1


In [3]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)

PFI_ROOT = Path(os.getenv("PFI_ROOT", "/content/drive/MyDrive/PFI_MVP"))
SPIDER_ROOT = Path(os.getenv("PFI_SPIDER_ROOT", str(PFI_ROOT / "data" / "SPIDER")))
RESULTS_ROOT = Path(os.getenv(
    "PFI_P10_7_RESULTS_ROOT",
    str(PFI_ROOT / "results" / "P10_7_spider_degenerative")
))
MODELS_ROOT = Path(os.getenv(
    "PFI_P10_7_MODELS_ROOT",
    str(PFI_ROOT / "models" / "P10_7_spider_degenerative")
))

def resolve_dir(base: Path, candidates: list[str]) -> Path:
    for rel in candidates:
        path = base / rel
        if path.is_dir():
            return path
    return base / candidates[0]

IMAGES_ROOT = Path(os.getenv(
    "PFI_SPIDER_IMAGES",
    str(resolve_dir(SPIDER_ROOT, ["images", "images/images"]))
))
MASKS_ROOT = Path(os.getenv(
    "PFI_SPIDER_MASKS",
    str(resolve_dir(SPIDER_ROOT, ["masks", "masks/masks"]))
))
GRADINGS_CSV = Path(os.getenv(
    "PFI_SPIDER_GRADINGS",
    str(SPIDER_ROOT / "radiological_gradings.csv")
))

SCOPE_PATH = RESULTS_ROOT / "task_scope_v1.json"
SPLIT_PATH = RESULTS_ROOT / "patient_split_v1.csv"
NB65_MARKER = RESULTS_ROOT / "NOTEBOOK_65_COMPLETE.json"
CROPS_ROOT = RESULTS_ROOT / "disc_crops_v1"
MANIFEST_PATH = RESULTS_ROOT / "disc_level_manifest_v1.csv"
QC_PATH = RESULTS_ROOT / "disc_level_dataset_qc_v1.json"
COMPLETE_PATH = RESULTS_ROOT / "NOTEBOOK_66_COMPLETE.json"

required = [SCOPE_PATH, SPLIT_PATH, NB65_MARKER, GRADINGS_CSV]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Faltan salidas de Notebook 65 o datos SPIDER: {missing}")
if not IMAGES_ROOT.is_dir() or not MASKS_ROOT.is_dir():
    raise FileNotFoundError(f"images/masks no disponibles: {IMAGES_ROOT}, {MASKS_ROOT}")

final_pt = [
    p for p in MODELS_ROOT.rglob("*.pt")
    if any(token in p.name.lower() for token in ("final", "frozen", "research_export"))
]
if final_pt:
    raise RuntimeError(f"P10.7 ya tiene un export final. Se aborta: {final_pt}")

scope = json.loads(SCOPE_PATH.read_text(encoding="utf-8"))
split_df = pd.read_csv(SPLIT_PATH, dtype={"Patient": str})
gradings = pd.read_csv(GRADINGS_CSV, dtype={"Patient": str})
print(scope["taskFamily"])
print(split_df["p10_7_split"].value_counts())


disc_level_degenerative_multitask
p10_7_split
dev_train        143
internal_test     39
dev_val           36
Name: count, dtype: int64


In [4]:
def atomic_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(tmp, path)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def canonical(value: Any) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip()).lower()

def encode_binary(value: Any) -> float:
    text = canonical(value)
    if text in {"", "nan", "na", "n/a", "missing", "unknown"}:
        return np.nan
    if text in {"0", "false", "no", "negative", "absent"}:
        return 0.0
    if text in {"1", "true", "yes", "positive", "present"}:
        return 1.0
    try:
        number = float(text)
        if number in {0.0, 1.0}:
            return number
    except ValueError:
        pass
    raise ValueError(f"Valor binario no reconocido: {value!r}")

def encode_pfirrmann(value: Any) -> float:
    text = canonical(value)
    if text == "":
        return np.nan
    match = re.search(r"[1-5]", text)
    if not match:
        raise ValueError(f"Grado Pfirrmann no reconocido: {value!r}")
    return float(int(match.group(0)) - 1)

def encode_modic(value: Any) -> float:
    text = canonical(value)
    if text in {"", "0", "none", "no", "absent"}:
        return 0.0
    normalized = text.replace("type", "").replace("tipo", "").strip()
    mapping = {
        "i": 1.0, "1": 1.0,
        "ii": 2.0, "2": 2.0,
        "iii": 3.0, "3": 3.0,
    }
    if normalized in mapping:
        return mapping[normalized]
    raise ValueError(f"Modic no reconocido: {value!r}")

label_encoders = {
    "pfirrmann_grade": ("Pfirrman grade", encode_pfirrmann),
    "modic_change": ("Modic", encode_modic),
    "upper_endplate_change": ("UP endplate", encode_binary),
    "lower_endplate_change": ("LOW endplate", encode_binary),
    "spondylolisthesis": ("Spondylolisthesis", encode_binary),
    "disc_herniation": ("Disc herniation", encode_binary),
    "disc_narrowing": ("Disc narrowing", encode_binary),
    "disc_bulging": ("Disc bulging", encode_binary),
}

for task_name, (column, encoder) in label_encoders.items():
    gradings[task_name] = gradings[column].map(encoder)


In [5]:
def parse_patient_and_sequence(path: Path) -> tuple[str, str] | None:
    stem = path.stem.lower()
    match = re.match(r"^(\d+)_(t1|t2|t2_space)$", stem)
    if not match:
        return None
    return match.group(1), match.group(2)

image_map: dict[tuple[str, str], Path] = {}
mask_map: dict[tuple[str, str], Path] = {}

for path in sorted(IMAGES_ROOT.rglob("*.mha")):
    parsed = parse_patient_and_sequence(path)
    if parsed:
        image_map[parsed] = path

for path in sorted(MASKS_ROOT.rglob("*.mha")):
    parsed = parse_patient_and_sequence(path)
    if parsed:
        mask_map[parsed] = path

def choose_sequence(patient: str, preferred: list[str]) -> tuple[Path, Path, str] | None:
    for sequence in preferred:
        image = image_map.get((patient, sequence))
        mask = mask_map.get((patient, sequence))
        if image and mask:
            return image, mask, sequence
    return None

inventory = []
for patient in sorted(split_df["Patient"].astype(str).unique()):
    inventory.append({
        "Patient": patient,
        "has_t1": choose_sequence(patient, ["t1"]) is not None,
        "has_t2": choose_sequence(patient, ["t2", "t2_space"]) is not None,
    })
inventory_df = pd.DataFrame(inventory)
print(inventory_df[["has_t1", "has_t2"]].value_counts())
display(inventory_df.head())


has_t1  has_t2
True    True      190
False   True       22
True    False       6
Name: count, dtype: int64


,Patient,has_t1,has_t2
0,1,True,True
1,10,True,True
2,100,True,True
3,101,True,True
4,104,True,True


In [6]:
def read_volume(path: Path) -> tuple[np.ndarray, tuple[float, float, float]]:
    image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(image).astype(np.float32)
    spacing_xyz = tuple(float(v) for v in image.GetSpacing())
    return array, spacing_xyz

def normalize_image(array: np.ndarray) -> np.ndarray:
    finite = array[np.isfinite(array)]
    if finite.size == 0:
        raise ValueError("Volumen sin valores finitos")
    lo, hi = np.percentile(finite, [1.0, 99.0])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(finite.min()), float(finite.max())
    if hi <= lo:
        return np.zeros_like(array, dtype=np.float32)
    return np.clip((array - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)

def crop_2p5d(
    image_path: Path,
    mask_path: Path,
    raw_disc_label: int,
    output_size: int = 224,
    margin_ratio: float = 0.80,
) -> tuple[np.ndarray, dict[str, Any]]:
    image, spacing = read_volume(image_path)
    mask, mask_spacing = read_volume(mask_path)
    if image.shape != mask.shape:
        raise ValueError(f"Shape incompatible image/mask: {image.shape} vs {mask.shape}")
    binary = np.isclose(mask, raw_disc_label)
    if not binary.any():
        raise ValueError(f"Label discal {raw_disc_label} ausente en {mask_path.name}")

    area_by_slice = binary.sum(axis=(1, 2))
    center_z = int(np.argmax(area_by_slice))
    ys, xs = np.where(binary[center_z])
    if ys.size == 0:
        raise ValueError("El corte de máxima área no contiene el disco")

    y0, y1 = int(ys.min()), int(ys.max()) + 1
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    height, width = y1 - y0, x1 - x0
    margin_y = max(8, int(round(height * margin_ratio)))
    margin_x = max(8, int(round(width * margin_ratio)))
    y0, y1 = max(0, y0 - margin_y), min(image.shape[1], y1 + margin_y)
    x0, x1 = max(0, x0 - margin_x), min(image.shape[2], x1 + margin_x)

    z_indices = [max(0, min(image.shape[0] - 1, center_z + delta)) for delta in (-1, 0, 1)]
    normalized = normalize_image(image)
    crop = normalized[z_indices, y0:y1, x0:x1]
    tensor = torch.from_numpy(crop).unsqueeze(0)
    resized = F.interpolate(
        tensor,
        size=(output_size, output_size),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0).numpy().astype(np.float32)

    metadata = {
        "sourceShape": list(image.shape),
        "spacingXyz": list(spacing),
        "centerSlice": center_z,
        "sliceIndices": z_indices,
        "bboxYx": [y0, y1, x0, x1],
        "rawDiscLabel": raw_disc_label,
    }
    return resized, metadata


In [7]:
# Se permite construir todos los crops, pero el contenido de etiquetas del internal_test
# no se resume ni se usa para decisiones. Solo se conserva para Notebook 68.
split_lookup = split_df.set_index("Patient")["p10_7_split"].to_dict()
CROPS_ROOT.mkdir(parents=True, exist_ok=True)

rows = []
failures = []

for record in gradings.to_dict("records"):
    patient = str(record["Patient"])
    split = split_lookup.get(patient)
    if split is None:
        failures.append({"Patient": patient, "reason": "split_missing"})
        continue

    try:
        ivd_label = int(float(record["IVD label"]))
    except Exception:
        failures.append({"Patient": patient, "reason": f"invalid_ivd_label:{record['IVD label']!r}"})
        continue

    raw_disc_label = 200 + ivd_label
    t1_source = choose_sequence(patient, ["t1"])
    t2_source = choose_sequence(patient, ["t2", "t2_space"])
    if t1_source is None and t2_source is None:
        failures.append({"Patient": patient, "IVD": ivd_label, "reason": "no_supported_sequence"})
        continue

    sample_hash = hashlib.sha256(f"{patient}|{ivd_label}".encode("utf-8")).hexdigest()[:16]
    sample_path = CROPS_ROOT / split / f"{sample_hash}.npz"
    sample_path.parent.mkdir(parents=True, exist_ok=True)

    t1 = np.zeros((3, 224, 224), dtype=np.float32)
    t2 = np.zeros((3, 224, 224), dtype=np.float32)
    t1_available = 0
    t2_available = 0
    crop_metadata: dict[str, Any] = {}

    try:
        if t1_source is not None:
            t1, meta = crop_2p5d(t1_source[0], t1_source[1], raw_disc_label)
            t1_available = 1
            crop_metadata["t1"] = meta
        if t2_source is not None:
            t2, meta = crop_2p5d(t2_source[0], t2_source[1], raw_disc_label)
            t2_available = 1
            crop_metadata["t2"] = meta
    except Exception as exc:
        failures.append({
            "Patient": patient,
            "IVD": ivd_label,
            "reason": f"{exc.__class__.__name__}:{exc}",
        })
        continue

    labels = {task: record[task] for task in label_encoders}
    label_values = np.array(
        [labels[task] if pd.notna(labels[task]) else -1.0 for task in label_encoders],
        dtype=np.float32,
    )
    label_mask = np.array(
        [1.0 if pd.notna(labels[task]) else 0.0 for task in label_encoders],
        dtype=np.float32,
    )

    np.savez_compressed(
        sample_path,
        t1=t1,
        t2=t2,
        availability=np.array([t1_available, t2_available], dtype=np.float32),
        ivd_label=np.array(ivd_label, dtype=np.int64),
        labels=label_values,
        label_mask=label_mask,
    )

    row = {
        "sample_id": sample_hash,
        "Patient": patient,
        "ivd_label": ivd_label,
        "raw_disc_label": raw_disc_label,
        "split": split,
        "crop_path": str(sample_path),
        "t1_available": t1_available,
        "t2_available": t2_available,
        "crop_metadata_json": json.dumps(crop_metadata, sort_keys=True),
    }
    row.update({
        task: (float(labels[task]) if pd.notna(labels[task]) else np.nan)
        for task in label_encoders
    })
    rows.append(row)

manifest = pd.DataFrame(rows).sort_values(["split", "Patient", "ivd_label"]).reset_index(drop=True)
failures_df = pd.DataFrame(failures)

if manifest.empty:
    raise RuntimeError("No se pudo construir ningún crop. Revisar labels y rutas.")

manifest.to_csv(MANIFEST_PATH, index=False)
failures_path = RESULTS_ROOT / "disc_level_crop_failures_v1.csv"
failures_df.to_csv(failures_path, index=False)

print(manifest["split"].value_counts())
print("samples:", len(manifest), "failures:", len(failures_df))
display(manifest.head())


split
dev_train        991
internal_test    272
dev_val          248
Name: count, dtype: int64
samples: 1511 failures: 9


,sample_id,Patient,ivd_label,raw_disc_label,split,crop_path,t1_available,t2_available,crop_metadata_json,pfirrmann_grade,modic_change,upper_endplate_change,lower_endplate_change,spondylolisthesis,disc_herniation,disc_narrowing,disc_bulging
0,3f22d6d875f4e163,1,1,201,dev_train,/content/drive/MyDrive/PFI_MVP/results/P10_7_s...,1,1,"{""t1"": {""bboxYx"": [165, 233, 4, 37], ""centerSl...",2.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,22074227d8462b39,1,2,202,dev_train,/content/drive/MyDrive/PFI_MVP/results/P10_7_s...,1,1,"{""t1"": {""bboxYx"": [125, 278, 7, 33], ""centerSl...",2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,6f49b6b637befada,1,3,203,dev_train,/content/drive/MyDrive/PFI_MVP/results/P10_7_s...,1,1,"{""t1"": {""bboxYx"": [123, 279, 16, 38], ""centerS...",2.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,0a538c0eabad742a,1,4,204,dev_train,/content/drive/MyDrive/PFI_MVP/results/P10_7_s...,1,1,"{""t1"": {""bboxYx"": [116, 301, 20, 44], ""centerS...",3.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
4,53c5785493bc683c,1,5,205,dev_train,/content/drive/MyDrive/PFI_MVP/results/P10_7_s...,1,1,"{""t1"": {""bboxYx"": [151, 291, 16, 48], ""centerS...",3.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [8]:
# QC de desarrollo únicamente: no se publican distribuciones del internal_test.
dev_manifest = manifest[manifest["split"].isin(["dev_train", "dev_val"])].copy()
task_qc = {}
for task in label_encoders:
    task_qc[task] = {
        "nonMissing": int(dev_manifest[task].notna().sum()),
        "distribution": {
            str(k): int(v)
            for k, v in dev_manifest[task].dropna().value_counts().sort_index().items()
        },
    }

qc = {
    "schemaVersion": "pfi.p10-7-disc-dataset-qc.v1",
    "status": "DISC_LEVEL_DATASET_READY",
    "samplesTotal": int(len(manifest)),
    "samplesBySplit": {str(k): int(v) for k, v in manifest["split"].value_counts().items()},
    "failures": int(len(failures_df)),
    "taskQcDevelopmentOnly": task_qc,
    "internalTestLabelsInspectedForSelection": False,
    "officialHiddenTestAccessed": False,
    "manifestSha256": sha256_file(MANIFEST_PATH),
}
atomic_json(QC_PATH, qc)
atomic_json(COMPLETE_PATH, {
    "status": "NOTEBOOK_66_COMPLETE",
    "manifestSha256": sha256_file(MANIFEST_PATH),
    "internalTestUsedForSelection": False,
})

print(json.dumps(qc, indent=2, ensure_ascii=False))
print("NOTEBOOK_66_COMPLETE")


{
  "schemaVersion": "pfi.p10-7-disc-dataset-qc.v1",
  "status": "DISC_LEVEL_DATASET_READY",
  "samplesTotal": 1511,
  "samplesBySplit": {
    "dev_train": 991,
    "internal_test": 272,
    "dev_val": 248
  },
  "failures": 9,
  "taskQcDevelopmentOnly": {
    "pfirrmann_grade": {
      "nonMissing": 1239,
      "distribution": {
        "0.0": 234,
        "1.0": 305,
        "2.0": 343,
        "3.0": 214,
        "4.0": 143
      }
    },
    "modic_change": {
      "nonMissing": 1239,
      "distribution": {
        "0.0": 853,
        "1.0": 4,
        "2.0": 376,
        "3.0": 6
      }
    },
    "upper_endplate_change": {
      "nonMissing": 1239,
      "distribution": {
        "0.0": 767,
        "1.0": 472
      }
    },
    "lower_endplate_change": {
      "nonMissing": 1239,
      "distribution": {
        "0.0": 762,
        "1.0": 477
      }
    },
    "spondylolisthesis": {
      "nonMissing": 1239,
      "distribution": {
        "0.0": 1206,
        "1.0": 33
      

In [9]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/PFI_MVP/results/"
    "P10_7_spider_degenerative"
)

MANIFEST_PATH = ROOT / "disc_level_manifest_v1.csv"
FAILURES_PATH = ROOT / "disc_level_crop_failures_v1.csv"
QC_PATH = ROOT / "disc_level_dataset_qc_v1.json"
COMPLETE_PATH = ROOT / "NOTEBOOK_66_COMPLETE.json"
CROPS_ROOT = ROOT / "disc_crops_v1"

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = pd.read_csv(MANIFEST_PATH, dtype={"Patient": str})
failures = pd.read_csv(FAILURES_PATH, dtype={"Patient": str})
qc = json.loads(QC_PATH.read_text(encoding="utf-8"))
complete = json.loads(COMPLETE_PATH.read_text(encoding="utf-8"))

physical_crops = list(CROPS_ROOT.rglob("*.npz"))
missing_paths = [
    path
    for path in manifest["crop_path"].astype(str)
    if not Path(path).is_file()
]

print("=== AUDITORÍA NOTEBOOK 66 ===")
print("status:", complete.get("status"))
print("manifest rows:", len(manifest))
print("physical crops:", len(physical_crops))
print("failures:", len(failures))

print("\nSamples por split:")
print(manifest["split"].value_counts().to_string())

print("\nFallos por split:")
failure_split = failures["Patient"].map(
    pd.read_csv(
        ROOT / "patient_split_v1.csv",
        dtype={"Patient": str}
    ).set_index("Patient")["p10_7_split"]
)
print(failure_split.value_counts(dropna=False).to_string())

print("\nMotivos de fallo:")
print(failures["reason"].value_counts().to_string())

print("\nIntegridad:")
print("sample_id duplicados:", int(manifest["sample_id"].duplicated().sum()))
print("crop_path duplicados:", int(manifest["crop_path"].duplicated().sum()))
print("crop paths faltantes:", len(missing_paths))

actual_sha = sha256_file(MANIFEST_PATH)
print("\nSHA actual:", actual_sha)
print("SHA QC:", qc.get("manifestSha256"))
print("SHA COMPLETE:", complete.get("manifestSha256"))
print(
    "SHA_MATCH:",
    actual_sha
    == qc.get("manifestSha256")
    == complete.get("manifestSha256")
)

=== AUDITORÍA NOTEBOOK 66 ===
status: NOTEBOOK_66_COMPLETE
manifest rows: 1511
physical crops: 1511
failures: 9

Samples por split:
split
dev_train        991
internal_test    272
dev_val          248

Fallos por split:
Patient
internal_test    8
dev_train        1

Motivos de fallo:
reason
ValueError:Label discal 204 ausente en 35_t2.mha     1
ValueError:Label discal 205 ausente en 35_t2.mha     1
ValueError:Label discal 206 ausente en 35_t2.mha     1
ValueError:Label discal 207 ausente en 35_t2.mha     1
ValueError:Label discal 208 ausente en 35_t2.mha     1
ValueError:Label discal 209 ausente en 35_t2.mha     1
ValueError:Label discal 207 ausente en 64_t2.mha     1
ValueError:Label discal 200 ausente en 107_t1.mha    1
ValueError:Label discal 200 ausente en 152_t1.mha    1

Integridad:
sample_id duplicados: 0
crop_path duplicados: 0
crop paths faltantes: 0

SHA actual: a3c5cb7fdc02dc2b3cf8836c7a379ca738a144f4a74fcc35d6dbb893de41efbb
SHA QC: a3c5cb7fdc02dc2b3cf8836c7a379ca738a144f4a7

In [11]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import SimpleITK as sitk

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
SPIDER_ROOT = PFI_ROOT / "data" / "SPIDER"
RESULTS_ROOT = PFI_ROOT / "results" / "P10_7_spider_degenerative"

def resolve_dir(base: Path, candidates: list[str]) -> Path:
    for relative in candidates:
        path = base / relative
        if path.is_dir():
            return path
    return base / candidates[0]

IMAGES_ROOT = resolve_dir(
    SPIDER_ROOT,
    ["images", "images/images"],
)
MASKS_ROOT = resolve_dir(
    SPIDER_ROOT,
    ["masks", "masks/masks"],
)

FAILURES_PATH = RESULTS_ROOT / "disc_level_crop_failures_v1.csv"
failures = pd.read_csv(FAILURES_PATH, dtype={"Patient": str})

def parse_patient_and_sequence(path: Path):
    stem = path.stem.lower()
    match = re.match(r"^(\d+)_(t1|t2|t2_space)$", stem)
    if not match:
        return None
    return match.group(1), match.group(2)

image_map = {}
mask_map = {}

for path in sorted(IMAGES_ROOT.rglob("*.mha")):
    parsed = parse_patient_and_sequence(path)
    if parsed:
        image_map[parsed] = path

for path in sorted(MASKS_ROOT.rglob("*.mha")):
    parsed = parse_patient_and_sequence(path)
    if parsed:
        mask_map[parsed] = path

print("Imágenes indexadas:", len(image_map))
print("Máscaras indexadas:", len(mask_map))

def find_pair(patient: str, candidates: list[str]):
    for sequence in candidates:
        image = image_map.get((patient, sequence))
        mask = mask_map.get((patient, sequence))
        if image is not None and mask is not None:
            return sequence, image, mask
    return None

rows = []

for failure in failures.to_dict("records"):
    patient = str(failure["Patient"])
    ivd_label = int(float(failure["IVD"]))
    raw_label = 200 + ivd_label

    for modality, candidates in {
        "t1": ["t1"],
        "t2": ["t2", "t2_space"],
    }.items():
        pair = find_pair(patient, candidates)

        if pair is None:
            rows.append({
                "Patient": patient,
                "IVD": ivd_label,
                "raw_label": raw_label,
                "modality": modality,
                "sequence": None,
                "pair_available": False,
                "target_present": False,
                "target_voxels": 0,
            })
            continue

        sequence, image_path, mask_path = pair
        mask = sitk.GetArrayFromImage(
            sitk.ReadImage(str(mask_path))
        )

        target_voxels = int(np.isclose(mask, raw_label).sum())

        rows.append({
            "Patient": patient,
            "IVD": ivd_label,
            "raw_label": raw_label,
            "modality": modality,
            "sequence": sequence,
            "pair_available": True,
            "target_present": target_voxels > 0,
            "target_voxels": target_voxels,
            "mask_file": mask_path.name,
        })

audit = pd.DataFrame(rows).sort_values(
    ["Patient", "IVD", "modality"]
)

display(audit)

recoverable = (
    audit.groupby(["Patient", "IVD"])["target_present"]
    .any()
)

print("\nResumen:")
print(
    recoverable.value_counts()
    .rename({
        True: "recuperable_con_al_menos_una_secuencia",
        False: "no_recuperable",
    })
    .to_string()
)

Imágenes indexadas: 447
Máscaras indexadas: 447


,Patient,IVD,raw_label,modality,sequence,pair_available,target_present,target_voxels,mask_file
14,107,0,200,t1,t1,True,False,0,107_t1.mha
15,107,0,200,t2,t2,True,False,0,107_t2.mha
16,152,0,200,t1,t1,True,False,0,152_t1.mha
17,152,0,200,t2,t2,True,False,0,152_t2.mha
0,35,4,204,t1,t1,True,True,18413,35_t1.mha
1,35,4,204,t2,t2,True,False,0,35_t2.mha
2,35,5,205,t1,t1,True,True,13660,35_t1.mha
3,35,5,205,t2,t2,True,False,0,35_t2.mha
4,35,6,206,t1,t1,True,True,9379,35_t1.mha
5,35,6,206,t2,t2,True,False,0,35_t2.mha



Resumen:
target_present
recuperable_con_al_menos_una_secuencia    7
no_recuperable                            2
